# NFL ATS Weekly Prediction Pipeline

Run via **papermill** from the project root:
```bash
papermill betting/predict_betting.ipynb /tmp/out.ipynb -p MODE thursday
```
**Modes:** `tuesday` · `thursday` · `sunday` · `backfill`

Or run all cells interactively — set `MODE` (and optionally `TARGET_WEEK`) in the Parameters cell.


## Parameters

Set `MODE` before running. Papermill overrides this parameter automatically:

| Mode | When | What it does |
|------|------|--------------|
| `tuesday` | Tue 9am ET | Updates previous week results, then runs new predictions |
| `thursday` | Thu 9pm ET | Refreshes predictions with latest injury reports |
| `sunday` | Sun 9am ET | Final predictions before kickoff |
| `backfill` | Manual | Runs predictions for `TARGET_WEEK`, then immediately fills in actual results |

`TARGET_WEEK` — override week auto-detection (required for `backfill`; optional for other modes).


In [ ]:
MODE        = "thursday"  # tuesday | thursday | sunday | backfill
TARGET_WEEK = None         # set to an int to override auto-detection (backfill)
TARGET_SEASON = None       # None = auto-detect from current date; override to lock a season


## Setup — Imports

Standard library imports. `FinalCfg` (defined in the Configuration cell below) must exist before `joblib.load()` — it is embedded in the XGBoost pkl and must be importable at deserialisation time.


In [ ]:
import os
import joblib
import lightgbm
import re as _re
import unicodedata as _ud
import numpy as np
import pandas as pd
import nflreadpy as nfl
from datetime import datetime
from dataclasses import dataclass, field
from typing import Tuple, Dict, Any
from pathlib import Path


## Configuration & Paths — Model Loading

Detects the working directory (project root vs `betting/` subdirectory) and resolves all file paths. Defines the `FinalCfg` dataclass (required by the XGBoost pkl), then loads all four production models:

- **Ensemble fixed75** (`ensemble_prod_model.pkl`) — primary edge-setter (0.75 XGB + 0.25 Ridge). Provides `ens_ridge`, `ens_scaler`, and `ens_enc` shared by Ridge and LightGBM.
- **XGBoost** (`xgboost_prod_model.pkl`) — sklearn pipeline (preprocessor + regressor); direction voter.
- **Ridge** (extracted from `ensemble_prod_model.pkl` as `ens_ridge`) — direction voter; uses shared `ens_scaler` and `ens_enc`.
- **LightGBM** (`lgbm_prod_model.pkl`) — independent direction voter.

Also loads `nfl_allpro_1997_2025.csv` and normalises historical team abbreviations via `TEAM_MAP`.


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
# Works whether kernel starts from project root or from betting/ directly
_cwd = Path.cwd()
_DIR = _cwd if _cwd.name == "betting" else _cwd / "betting"
_MODELS_DIR = _DIR / "models"

TRACKER_PATH    = str(_DIR / "predictions_tracker.csv")
ALLPRO_CSV_PATH = str(_DIR / "nfl_allpro_1997_2025.csv")
XGB_MODEL_PATH  = str(_MODELS_DIR / "xgboost_prod_model.pkl")
ENS_MODEL_PATH  = str(_MODELS_DIR / "ensemble_prod_model.pkl")
LGBM_MODEL_PATH = str(_MODELS_DIR / "lgbm_prod_model.pkl")

print(f"Running in {MODE.upper()} mode — {datetime.now().strftime('%Y-%m-%d %H:%M')}")

# ── Load XGBoost production model ─────────────────────────────────────────────
@dataclass(frozen=True)
class FinalCfg:
    test_size: float = 0.2
    random_state: int = 42
    oof_splits: int = 5
    weight_win: float = 2.0
    weight_loss: float = 1.0
    drop_non_features: Tuple[str, ...] = ('game_id', 'home_team', 'away_team', 'season', 'week')
    categorical_cols: Tuple[str, ...] = ('roof', 'surface')
    boolean_cols: Tuple[str, ...] = (
        'is_playoff', 'is_final_week', 'home_qb_switch', 'away_qb_switch',
        'is_home_qb_new', 'is_away_qb_new'
    )
    base_xgb_params: Dict[str, Any] = field(default_factory=lambda: dict(
        n_estimators=500, max_depth=3, learning_rate=0.01, min_child_weight=3,
        subsample=0.6, colsample_bytree=0.6, reg_alpha=1.0, reg_lambda=3.0,
        objective='reg:squarederror', random_state=42, tree_method='hist', n_jobs=1
    ))

xgb_res      = joblib.load(XGB_MODEL_PATH)
pipeline     = xgb_res['pipeline']
pre          = pipeline.named_steps['preprocessor']
_known_cats  = set(FinalCfg().categorical_cols)
cat_cols, num_cols = None, None
for _, _, _cols in pre.transformers_:
    if set(_cols) & _known_cats:
        cat_cols = list(_cols)
    else:
        num_cols = list(_cols)
assert cat_cols is not None and num_cols is not None, "Could not identify categorical/numeric transformer slots"
model_features = cat_cols + num_cols
print(f"XGBoost loaded — {len(model_features)} features")


# ── Load Ensemble (fixed75) ───────────────────────────────────────────────────
ens_pkg        = joblib.load(ENS_MODEL_PATH)
ens_xgb        = ens_pkg["xgb_model"]
ens_ridge      = ens_pkg["ridge_model"]
ens_scaler     = ens_pkg["scaler"]
ens_xgb_weight = ens_pkg["xgb_weight"]
ens_feat_cols  = ens_pkg["feature_cols"]
ens_enc        = ens_pkg["roof_surface_encoder"]
print(f"Ensemble loaded — XGB weight={ens_xgb_weight}")

# ── Load LightGBM voter ──────────────────────────────────────────────────────
lgbm_pkg       = joblib.load(LGBM_MODEL_PATH)
lgbm_model     = lgbm_pkg["model"]
lgbm_feat_cols = lgbm_pkg["feature_cols"]
print(f"LightGBM loaded — {len(lgbm_feat_cols)} features")

# ── Load static data ──────────────────────────────────────────────────────────
allpro_df = pd.read_csv(ALLPRO_CSV_PATH)
allpro_df = allpro_df[allpro_df["Team"] != "2TM"].copy()

TEAM_MAP = {
    "STL": "LA", "LAR": "LA", "OAK": "LV", "LVR": "LV",
    "SD": "LAC", "SDG": "LAC", "NWE": "NE", "KAN": "KC",
    "GNB": "GB", "NOR": "NO", "TAM": "TB", "SFO": "SF",
    # Pre-2002 / alternate abbreviations present in historical AllPro CSV
    "ARZ": "ARI", "BLT": "BAL", "CLV": "CLE", "HST": "HOU", "JAC": "JAX",
}
allpro_df["Team"] = allpro_df["Team"].replace(TEAM_MAP)


## Helper Functions

**`get_week_info(season)`** — Detects the next unplayed week and the previous completed week from the schedule.

**`build_features(target_week, target_season, full_schedule, pbp_rp, allpro_df, week_margin_lkp)`** — Builds the 77-feature matrix for a target week using only data available before that week. Feature groups:

| Group | Features |
|-------|----------|
| 1 — Schedule context | `roof`, `surface`, `is_playoff`, `is_final_week` |
| 2 — Rolling PBP | EPA, yards/play, play count (5-game windows); home/away offense & defense diffs |
| 3 — Strength of schedule | Opponent win% (rolling 3-game + season-long); `sos_diff`, `season_sos_diff` |
| 4 — All-Pro roster quality | Weighted 3-year lookback (4/2/1 weights); offense/defense split; prev-year counts |
| 5 — Rolling performance | Win%, points scored/allowed, cover rate (5-game windows); `league_rolling_avg_abs_margin_by_week` |
| 6 — Situational PBP | Sacks, turnovers, third-down conversion rate (5-game windows); home/away diffs |
| 7 — QB switch | `home_qb_switch`, `away_qb_switch`, `is_home_qb_new`, `is_away_qb_new` |
| 8 — Passer rating | Prior-season NFL passer rating for starting QB; `diff_qbr_prev_year` |
| 9 — Injuries | Out-player count; All-Pro-weighted injury impact; `diff_active_allpro_weighted` |
| 10 — Coach win% | Career win% prior to each game for home/away coach; loaded from 1999–present schedule |

**`build_numeric_features(upcoming_df, feature_cols, enc)`** — Ordinal-encodes `roof`/`surface` and returns a float32 matrix for the Ensemble and LightGBM models (which don't use the sklearn pipeline).

**`run_predictions(target_week, ...)`** — Runs Ensemble fixed75 (edge-setter and sort key) and three direction voters (XGBoost standalone, Ridge, LightGBM). Computes per-model edge vs. Vegas spread, assigns `consensus_tier`, and sorts games by absolute ensemble edge.

**`update_results(season, week)`** — After games are played, fetches actual scores from `nflreadpy`, fills `actual_margin` / `home_covered` / `model_correct` in the tracker CSV.

**`log_predictions(results_df, season, week, mode)`** — Appends (or replaces on refresh) a week's predictions in `betting/predictions_tracker.csv`.


In [ ]:
def _norm_name(s):
    if not isinstance(s, str): return ''
    s = ''.join(c for c in _ud.normalize('NFD', s) if _ud.category(c) != 'Mn')
    s = s.lower().strip()
    s = _re.sub(r'\s+(jr\.?|sr\.?|ii|iii|iv|v)\s*$', '', s)
    s = _re.sub(r"[\'.\-]", \'\', s)
    return _re.sub(r'\s+', ' ', s).strip()


# ── Helper: detect current/previous week ─────────────────────────────────────
def get_week_info(season, schedule_df=None):
    if schedule_df is None:
        raw      = nfl.load_schedules([season])
        schedule = raw.to_pandas() if hasattr(raw, 'to_pandas') else pd.DataFrame(raw)
    else:
        schedule = schedule_df
    reg      = schedule[(schedule['season'] == season) & (schedule['game_type'] == 'REG')]
    future   = reg[reg['result'].isna()]
    done     = reg[reg['result'].notna()]
    if future.empty:
        return None, int(done['week'].max())
    upcoming_week = int(future['week'].min())
    prev_week     = upcoming_week - 1 if upcoming_week > 1 else None
    return upcoming_week, prev_week

# ── Full feature pipeline ─────────────────────────────────────────────────────
def build_features(target_week, target_season, full_schedule, pbp_rp, allpro_df, week_margin_lkp=None, coach_hist_df=None):
    """
    Builds all 77 features for target_week using only data
    available before that week. Returns upcoming DataFrame.
    """
    history  = full_schedule[
        (full_schedule['season'] == target_season) &
        (full_schedule['week']   <  target_week) &
        (full_schedule['result'].notna())
    ].copy()

    upcoming = full_schedule[
        (full_schedule['season'] == target_season) &
        (full_schedule['week']   == target_week)
    ].copy()

    if upcoming.empty:
        return None

    if history.empty:
        print(f"  ℹ️  Week 1: no season history yet — SOS/scoring/cover features will be zero-filled")

    # ── Group 1 — schedule features ───────────────────────────────────────────
    upcoming['is_playoff']  = (upcoming['game_type'] != 'REG')
    final_week_num          = full_schedule[full_schedule['game_type'] == 'REG']['week'].max()
    upcoming['is_final_week'] = (
        (upcoming['game_type'] == 'REG') & (upcoming['week'] == final_week_num)
    )

    if upcoming['is_playoff'].any():
        print(f"  ⚠️  Week {target_week} contains playoff games — models were trained on REG season only")

    # ── Group 2 — rolling PBP stats ──────────────────────────────────────────
    pbp_s      = pbp_rp[
        ((pbp_rp['season'] == target_season) & (pbp_rp['week'] < target_week)) |
        (pbp_rp['season'] == target_season - 1)
    ].copy()
    wk_lookup  = pbp_rp[['game_id', 'week', 'season']].drop_duplicates()

    off_stats = (
        pbp_s.groupby(['game_id', 'posteam'])
        .agg(avg_epa=('epa', 'mean'), avg_yards=('yards_gained', 'mean'), play_count=('play_id', 'count'))
        .reset_index().rename(columns={'posteam': 'team'})
    )
    off_stats = off_stats.merge(wk_lookup[['game_id','week','season']], on='game_id', how='left')
    off_stats = off_stats.sort_values(['team', 'season', 'week'])
    for feat in ['avg_epa', 'avg_yards', 'play_count']:
        off_stats[f'rolling_{feat}'] = (
            off_stats.groupby('team')[feat]
            .transform(lambda x: x.rolling(5, min_periods=1).mean())
        )

    def_stats = (
        pbp_s.groupby(['game_id', 'defteam'])
        .agg(allowed_avg_epa=('epa', 'mean'), allowed_avg_yards=('yards_gained', 'mean'), allowed_play_count=('play_id', 'count'))
        .reset_index().rename(columns={'defteam': 'team'})
    )
    def_stats = def_stats.merge(wk_lookup[['game_id','week','season']], on='game_id', how='left')
    def_stats = def_stats.sort_values(['team', 'season', 'week'])
    for feat in ['allowed_avg_epa', 'allowed_avg_yards', 'allowed_play_count']:
        def_stats[f'rolling_{feat}'] = (
            def_stats.groupby('team')[feat]
            .transform(lambda x: x.rolling(5, min_periods=1).mean())
        )

    latest_off = off_stats.groupby('team').last().reset_index()
    latest_def = def_stats.groupby('team').last().reset_index()

    for side, df in [('home_team', latest_off), ('away_team', latest_off)]:
        prefix = 'home_' if side == 'home_team' else 'away_'
        cols   = [c for c in df.columns if c.startswith('rolling_')]
        upcoming = upcoming.merge(
            df[['team'] + cols].rename(columns={'team': side, **{c: f'{prefix}{c}' for c in cols}}),
            on=side, how='left'
        )
    for side, df in [('home_team', latest_def), ('away_team', latest_def)]:
        prefix = 'home_' if side == 'home_team' else 'away_'
        cols   = [c for c in df.columns if c.startswith('rolling_')]
        upcoming = upcoming.merge(
            df[['team'] + cols].rename(columns={'team': side, **{c: f'{prefix}{c}' for c in cols}}),
            on=side, how='left'
        )

    upcoming['epa_home_off_away_def_rolling_diff']       = upcoming['home_rolling_avg_epa']           - upcoming['away_rolling_allowed_avg_epa']
    upcoming['epa_home_def_away_off_rolling_diff']       = upcoming['home_rolling_allowed_avg_epa']   - upcoming['away_rolling_avg_epa']
    upcoming['avg_yards_home_off_away_def_rolling_diff'] = upcoming['home_rolling_avg_yards']         - upcoming['away_rolling_allowed_avg_yards']
    upcoming['avg_yards_home_def_away_off_rolling_diff'] = upcoming['home_rolling_allowed_avg_yards'] - upcoming['away_rolling_avg_yards']
    upcoming['play_count_home_off_away_def_rolling_diff']= upcoming['home_rolling_play_count']        - upcoming['away_rolling_allowed_play_count']
    upcoming['play_count_home_def_away_off_rolling_diff']= upcoming['home_rolling_allowed_play_count']- upcoming['away_rolling_play_count']

    # ── Group 3 — SOS ────────────────────────────────────────────────────────
    _req_cols = ['season', 'week', 'home_team', 'away_team', 'home_score', 'away_score', 'result', 'spread_line']
    if coach_hist_df is not None and all(c in coach_hist_df.columns for c in _req_cols):
        _hist_rolling = coach_hist_df[
            ((coach_hist_df['season'] == target_season) & (coach_hist_df['week'] < target_week)) |
            (coach_hist_df['season'] == target_season - 1)
        ][_req_cols].copy()
    else:
        _hist_rolling = history[[c for c in _req_cols if c in history.columns]].copy()

    home_g = _hist_rolling[['season','week','home_team','away_team','home_score','away_score']].copy()
    home_g.columns = ['season','week','team','opponent','team_score','opp_score']
    away_g = _hist_rolling[['season','week','away_team','home_team','away_score','home_score']].copy()
    away_g.columns = ['season','week','team','opponent','team_score','opp_score']
    long_df = pd.concat([home_g, away_g]).sort_values(['team','season','week'])
    long_df['team_win'] = (long_df['team_score'] > long_df['opp_score']).astype(int)
    long_df['win_pct']  = long_df.groupby('team')['team_win'].transform(
        lambda x: x.shift(1).expanding().mean()
    )
    opp_wp = long_df[['season','week','team','win_pct']].copy()
    opp_wp.columns = ['season','week','opponent','opponent_win_pct']
    long_df = long_df.merge(opp_wp, on=['season','week','opponent'], how='left')
    long_df['recent_sos'] = long_df.groupby('team')['opponent_win_pct'].transform(
        lambda x: x.rolling(3, min_periods=1).mean().fillna(0)
    )
    long_df['season_sos'] = long_df.groupby('team')['opponent_win_pct'].transform(
        lambda x: x.expanding().mean().fillna(0)
    )
    latest_sos = long_df.groupby('team').last().reset_index()[['team','recent_sos','season_sos']]
    upcoming = upcoming.merge(latest_sos.rename(columns={'team':'home_team','recent_sos':'home_recent_sos_opponent_avg','season_sos':'home_season_sos_opponent_avg'}), on='home_team', how='left')
    upcoming = upcoming.merge(latest_sos.rename(columns={'team':'away_team','recent_sos':'away_recent_sos_opponent_avg','season_sos':'away_season_sos_opponent_avg'}), on='away_team', how='left')
    for col in ['home_recent_sos_opponent_avg','home_season_sos_opponent_avg','away_recent_sos_opponent_avg','away_season_sos_opponent_avg']:
        upcoming[col] = upcoming[col].fillna(0)
    upcoming['sos_diff']        = upcoming['home_recent_sos_opponent_avg'] - upcoming['away_recent_sos_opponent_avg']
    upcoming['season_sos_diff'] = upcoming['home_season_sos_opponent_avg'] - upcoming['away_season_sos_opponent_avg']

    # ── Group 4 — All-Pro ────────────────────────────────────────────────────
    offense_df = allpro_df[allpro_df['Side'] == 'offense'].copy()
    defense_df = allpro_df[allpro_df['Side'] == 'defense'].copy()

    def build_weighted(df_ap):
        frames = []
        for year in range(2006, target_season + 1):
            curr = []
            for yrs_back, weight in zip([1, 2, 3], [4, 2, 1]):
                tmp = df_ap[df_ap['Year'] == year - yrs_back].copy()
                tmp['Weight'] = weight
                tmp['Year_target'] = year
                curr.append(tmp)
            comb    = pd.concat(curr)
            deduped = comb.sort_values('Weight', ascending=False).drop_duplicates(['Player', 'Year_target'])
            wc      = deduped.groupby(['Year_target', 'Team'])['Weight'].sum().reset_index()
            wc.columns = ['season', 'Team', 'allpro_weighted']
            frames.append(wc)
        return pd.concat(frames, ignore_index=True)

    weighted_allpro  = build_weighted(allpro_df)
    offense_weighted = build_weighted(offense_df)
    defense_weighted = build_weighted(defense_df)

    def merge_allpro(df, feat_df, feat_col, home_col, away_col):
        lookup = feat_df[feat_df['season'] == target_season].drop(columns='season')
        df = df.merge(lookup.rename(columns={'Team': 'home_team', feat_col: home_col}), on='home_team', how='left')
        df = df.merge(lookup.rename(columns={'Team': 'away_team', feat_col: away_col}), on='away_team', how='left')
        df[home_col] = df[home_col].fillna(0)
        df[away_col] = df[away_col].fillna(0)
        return df

    upcoming = merge_allpro(upcoming, weighted_allpro,  'allpro_weighted', 'home_allpro_last_3_years_weighted', 'away_allpro_last_3_years_weighted')
    upcoming = merge_allpro(upcoming, offense_weighted, 'allpro_weighted', 'home_offense_allpro_3_years',        'away_offense_allpro_3_years')
    upcoming = merge_allpro(upcoming, defense_weighted, 'allpro_weighted', 'home_defense_allpro_3_years',        'away_defense_allpro_3_years')

    prev_overall = allpro_df.assign(season=allpro_df['Year']+1).groupby(['season','Team'])['Player'].nunique().reset_index(name='allpro_prev_year')
    prev_offense = offense_df.assign(season=offense_df['Year']+1).groupby(['season','Team'])['Player'].nunique().reset_index(name='allpro_prev_year')
    prev_defense = defense_df.assign(season=defense_df['Year']+1).groupby(['season','Team'])['Player'].nunique().reset_index(name='allpro_prev_year')

    upcoming = merge_allpro(upcoming, prev_overall, 'allpro_prev_year', 'home_allpro_prev_year',         'away_allpro_prev_year')
    upcoming = merge_allpro(upcoming, prev_offense, 'allpro_prev_year', 'home_offense_allpro_prev_year', 'away_offense_allpro_prev_year')
    upcoming = merge_allpro(upcoming, prev_defense, 'allpro_prev_year', 'home_defense_allpro_prev_year', 'away_defense_allpro_prev_year')

    upcoming['diff_allpro_last_3_years_weighted']        = upcoming['home_allpro_last_3_years_weighted']  - upcoming['away_allpro_last_3_years_weighted']
    upcoming['diff_allpro_prev_year']                    = upcoming['home_allpro_prev_year']               - upcoming['away_allpro_prev_year']
    upcoming['allpro_diff_home_off_away_def_3_years']    = upcoming['home_offense_allpro_3_years']         - upcoming['away_defense_allpro_3_years']
    upcoming['allpro_diff_home_def_away_off_3_years ']   = upcoming['home_defense_allpro_3_years']         - upcoming['away_offense_allpro_3_years']   # trailing space matches model
    upcoming['allpro_diff_home_off_away_def_prev_year']  = upcoming['home_offense_allpro_prev_year']       - upcoming['away_defense_allpro_prev_year']
    upcoming['allpro_diff_home_def_away_off_prev_year']  = upcoming['home_defense_allpro_prev_year']       - upcoming['away_offense_allpro_prev_year']

    # ── Group 5 — rolling win pct, scoring, cover rate, league margin ────────
    long_df['rolling_win_pct'] = long_df.groupby('team')['team_win'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )
    latest_wp = long_df.groupby('team').last().reset_index()[['team','rolling_win_pct']]
    upcoming  = upcoming.merge(latest_wp.rename(columns={'team':'home_team','rolling_win_pct':'home_rolling_win_pct'}), on='home_team', how='left')
    upcoming  = upcoming.merge(latest_wp.rename(columns={'team':'away_team','rolling_win_pct':'away_rolling_win_pct'}), on='away_team', how='left')

    long_df['rolling_scored']  = long_df.groupby('team')['team_score'].transform(lambda x: x.rolling(5, min_periods=1).mean())
    long_df['rolling_allowed'] = long_df.groupby('team')['opp_score'].transform(lambda x: x.rolling(5, min_periods=1).mean())
    latest_sc = long_df.groupby('team').last().reset_index()[['team','rolling_scored','rolling_allowed']]
    upcoming  = upcoming.merge(latest_sc.rename(columns={'team':'home_team','rolling_scored':'home_rolling_scored','rolling_allowed':'home_rolling_allowed'}), on='home_team', how='left')
    upcoming  = upcoming.merge(latest_sc.rename(columns={'team':'away_team','rolling_scored':'away_rolling_scored','rolling_allowed':'away_rolling_allowed'}), on='away_team', how='left')
    upcoming['scoring_diff']         = upcoming['home_rolling_scored'] - upcoming['away_rolling_scored']
    upcoming['scoring_diff_reverse'] = upcoming['away_rolling_scored'] - upcoming['home_rolling_scored']

    history2 = _hist_rolling.copy()
    # push=0 (home) / push=1 (away) matches training behavior — retrain before switching to NaN
    history2['home_covered'] = (history2['result'] > history2['spread_line']).astype(int)
    home_cov = history2[['season','week','home_team','home_covered']].rename(columns={'home_team':'team','home_covered':'covered'})
    away_cov = history2[['season','week','away_team','home_covered']].copy()
    away_cov['covered'] = 1 - away_cov['home_covered']
    away_cov = away_cov.drop(columns='home_covered').rename(columns={'away_team':'team'})
    cover_df = pd.concat([home_cov, away_cov]).sort_values(['team','season','week'])
    cover_df['rolling_cover_rate'] = cover_df.groupby('team')['covered'].transform(lambda x: x.rolling(5, min_periods=1).mean())
    latest_cover = cover_df.groupby('team').last().reset_index()[['team','rolling_cover_rate']]
    upcoming = upcoming.merge(latest_cover.rename(columns={'team':'home_team','rolling_cover_rate':'home_rolling_cover_rate'}), on='home_team', how='left')
    upcoming = upcoming.merge(latest_cover.rename(columns={'team':'away_team','rolling_cover_rate':'away_rolling_cover_rate'}), on='away_team', how='left')
    upcoming['cover_rate_diff'] = upcoming['home_rolling_cover_rate'] - upcoming['away_rolling_cover_rate']

    # Cross-season per-week avg absolute margin — matches training definition
    if week_margin_lkp is not None and target_week in week_margin_lkp.index:
        upcoming['league_rolling_avg_abs_margin_by_week'] = float(week_margin_lkp[target_week])
    elif week_margin_lkp is not None and len(week_margin_lkp) > 0:
        upcoming['league_rolling_avg_abs_margin_by_week'] = float(week_margin_lkp.mean())
    else:
        _wk_avgs = history.groupby('week')['result'].apply(lambda x: x.abs().mean()).sort_index()
        upcoming['league_rolling_avg_abs_margin_by_week'] = float(_wk_avgs.iloc[-1]) if len(_wk_avgs) > 0 else 0.0

    # ── Group 6 — sacks, turnovers, third down (from PBP) ───────────────────
    sack_df = pbp_s[pbp_s['sack'] == 1].copy()
    sacks   = sack_df.groupby(['game_id','defteam']).size().reset_index(name='sacks').rename(columns={'defteam':'team'})
    sacks   = sacks.merge(wk_lookup[['game_id','week','season']], on='game_id', how='left').sort_values(['team','season','week'])
    sacks['rolling_sacks'] = sacks.groupby('team')['sacks'].transform(lambda x: x.rolling(5, min_periods=1).mean())
    latest_sacks = sacks.groupby('team').last().reset_index()[['team','rolling_sacks']]
    upcoming = upcoming.merge(latest_sacks.rename(columns={'team':'home_team','rolling_sacks':'home_rolling_sacks'}), on='home_team', how='left')
    upcoming = upcoming.merge(latest_sacks.rename(columns={'team':'away_team','rolling_sacks':'away_rolling_sacks'}), on='away_team', how='left')
    upcoming['sack_diff']         = upcoming['home_rolling_sacks'] - upcoming['away_rolling_sacks']
    upcoming['sack_diff_reverse'] = upcoming['away_rolling_sacks'] - upcoming['home_rolling_sacks']

    pbp_s2 = pbp_s.copy()
    pbp_s2['turnover'] = ((pbp_s2['interception'] == 1) | (pbp_s2['fumble_lost'] == 1)).astype(int)
    to_df = pbp_s2.groupby(['game_id','posteam'])['turnover'].sum().reset_index().rename(columns={'posteam':'team'})
    to_df = to_df.merge(wk_lookup[['game_id','week','season']], on='game_id', how='left').sort_values(['team','season','week'])
    to_df['rolling_turnovers'] = to_df.groupby('team')['turnover'].transform(lambda x: x.rolling(5, min_periods=1).mean())
    latest_to = to_df.groupby('team').last().reset_index()[['team','rolling_turnovers']]
    upcoming  = upcoming.merge(latest_to.rename(columns={'team':'home_team','rolling_turnovers':'home_rolling_turnovers'}), on='home_team', how='left')
    upcoming  = upcoming.merge(latest_to.rename(columns={'team':'away_team','rolling_turnovers':'away_rolling_turnovers'}), on='away_team', how='left')
    upcoming['turnover_diff']         = upcoming['home_rolling_turnovers'] - upcoming['away_rolling_turnovers']
    upcoming['turnover_diff_reverse'] = upcoming['away_rolling_turnovers'] - upcoming['home_rolling_turnovers']

    pbp_s2['third_att']  = (pbp_s2['down'] == 3).astype(int)
    pbp_s2['third_conv'] = ((pbp_s2['down'] == 3) & (pbp_s2['first_down'] == 1)).astype(int)
    third_df = pbp_s2.groupby(['game_id','posteam']).agg(third_att=('third_att','sum'), third_conv=('third_conv','sum')).reset_index().rename(columns={'posteam':'team'})
    third_df['third_down_rate'] = third_df['third_conv'] / third_df['third_att'].replace(0, 1)
    third_df = third_df.merge(wk_lookup[['game_id','week','season']], on='game_id', how='left').sort_values(['team','season','week'])
    third_df['rolling_third'] = third_df.groupby('team')['third_down_rate'].transform(lambda x: x.rolling(5, min_periods=1).mean())
    latest_third = third_df.groupby('team').last().reset_index()[['team','rolling_third']]
    upcoming = upcoming.merge(latest_third.rename(columns={'team':'home_team','rolling_third':'home_rolling_third_down'}), on='home_team', how='left')
    upcoming = upcoming.merge(latest_third.rename(columns={'team':'away_team','rolling_third':'away_rolling_third_down'}), on='away_team', how='left')
    upcoming['third_down_diff']         = upcoming['home_rolling_third_down'] - upcoming['away_rolling_third_down']
    upcoming['third_down_diff_reverse'] = upcoming['away_rolling_third_down'] - upcoming['home_rolling_third_down']

    # ── Group 7 — QB switch ──────────────────────────────────────────────────
    # Include prior season so week-1 predictions compare against last year's final QB
    try:
        _prior_raw = nfl.load_schedules([target_season - 1])
        _prior_sched = _prior_raw.to_pandas() if hasattr(_prior_raw, 'to_pandas') else pd.DataFrame(_prior_raw)
        _prior_sched = _prior_sched[_prior_sched['result'].notna()].copy()
    except Exception:
        _prior_sched = pd.DataFrame(columns=list(history.columns))
    _qb_hist = pd.concat([
        _prior_sched[['season','week','home_team','away_team','home_qb_name','away_qb_name']],
        history[['season','week','home_team','away_team','home_qb_name','away_qb_name']]
    ]).dropna(subset=['home_team','away_team'])
    home_qbs = _qb_hist[['season','week','home_team','home_qb_name']].rename(columns={'home_team':'team','home_qb_name':'qb_name'})
    away_qbs = _qb_hist[['season','week','away_team','away_qb_name']].rename(columns={'away_team':'team','away_qb_name':'qb_name'})
    team_qbs = pd.concat([home_qbs, away_qbs]).sort_values(['team','season','week'])
    last_qb  = team_qbs.groupby('team').last().reset_index()[['team','qb_name']].rename(columns={'qb_name':'last_qb'})
    upcoming = upcoming.merge(last_qb.rename(columns={'team':'home_team','last_qb':'home_last_qb'}), on='home_team', how='left')
    upcoming = upcoming.merge(last_qb.rename(columns={'team':'away_team','last_qb':'away_last_qb'}), on='away_team', how='left')
    upcoming['home_qb_switch'] = (
        upcoming['home_qb_name'].notna() &
        upcoming['home_last_qb'].notna() &
        (upcoming['home_qb_name'] != upcoming['home_last_qb'])
    )
    upcoming['away_qb_switch'] = (
        upcoming['away_qb_name'].notna() &
        upcoming['away_last_qb'].notna() &
        (upcoming['away_qb_name'] != upcoming['away_last_qb'])
    )
    # Both pairs are currently synonymous; differentiate only if models are retrained
    upcoming['is_home_qb_new'] = upcoming['home_qb_switch']
    upcoming['is_away_qb_new'] = upcoming['away_qb_switch']

    # ── Group 8 — passer rating (prior season) ───────────────────────────────
    pass_plays = pbp_rp[
        (pbp_rp['play_type'] == 'pass') &
        (pbp_rp['season']    == target_season - 1) &
        (pbp_rp['passer_player_name'].notna())
    ].copy()
    qb_stats = pass_plays.groupby(['season','posteam','passer_player_name']).agg(
        attempts=('pass_attempt','sum'), completions=('complete_pass','sum'),
        yards=('passing_yards','sum'), tds=('pass_touchdown','sum'), ints=('interception','sum')
    ).reset_index()
    qb_stats = qb_stats[qb_stats['attempts'] >= 100]

    def passer_rating(row):
        a = max(0, min(((row['completions']/row['attempts']) - 0.3) * 5,   2.375))
        b = max(0, min(((row['yards']/row['attempts']) - 3) * 0.25,        2.375))
        c = max(0, min(row['tds']/row['attempts'] * 20,                    2.375))
        d = max(0, min(2.375 - (row['ints']/row['attempts'] * 25),         2.375))
        return ((a+b+c+d)/6)*100

    qb_stats['passer_rating'] = qb_stats.apply(passer_rating, axis=1)
    starter_qb = qb_stats.sort_values('attempts', ascending=False).groupby(['season','posteam']).first().reset_index()[['season','posteam','passer_rating']]
    pr_prev    = starter_qb[starter_qb['season'] == target_season - 1][['posteam','passer_rating']]
    median_pr = pr_prev['passer_rating'].median()
    upcoming = upcoming.merge(pr_prev.rename(columns={'posteam':'home_team','passer_rating':'home_qbr_prev_year'}), on='home_team', how='left')
    upcoming = upcoming.merge(pr_prev.rename(columns={'posteam':'away_team','passer_rating':'away_qbr_prev_year'}), on='away_team', how='left')
    upcoming['home_qbr_prev_year'] = upcoming['home_qbr_prev_year'].fillna(median_pr)
    upcoming['away_qbr_prev_year'] = upcoming['away_qbr_prev_year'].fillna(median_pr)
    upcoming['diff_qbr_prev_year'] = upcoming['home_qbr_prev_year'] - upcoming['away_qbr_prev_year']

    # ── Group 9 — injuries ───────────────────────────────────────────────────
    try:
        raw_inj = nfl.load_injuries(seasons=[target_season])
        inj_df  = raw_inj.to_pandas() if hasattr(raw_inj, 'to_pandas') else pd.DataFrame(raw_inj)
        _STATUS_WEIGHT = {'Out': 1.0, 'Doubtful': 0.75}
        inj_week = inj_df[(inj_df['report_status'].isin(['Out', 'Doubtful'])) & (inj_df['week'] == target_week)].copy()
        inj_week['_status_wt'] = inj_week['report_status'].map(_STATUS_WEIGHT).fillna(0)
        inj_by_team = inj_week.groupby('team')['_status_wt'].sum().reset_index().rename(columns={'_status_wt': 'count'})

        # Basic count
        upcoming = upcoming.merge(inj_by_team.rename(columns={'team':'home_team','count':'home_injured_count'}), on='home_team', how='left')
        upcoming = upcoming.merge(inj_by_team.rename(columns={'team':'away_team','count':'away_injured_count'}), on='away_team', how='left')
        upcoming[['home_injured_count','away_injured_count']] = upcoming[['home_injured_count','away_injured_count']].fillna(0)
        upcoming['diff_injured_count'] = upcoming['home_injured_count'] - upcoming['away_injured_count']

        # Injured allpro weighted
        inj_all = inj_df[(inj_df['report_status'].isin(['Out', 'Doubtful'])) & (inj_df['week'] == target_week)].copy()
        inj_all['_status_wt'] = inj_all['report_status'].map(_STATUS_WEIGHT).fillna(0)
        inj_all['season'] = inj_all['season'].astype(int)
        allpro_hist = []
        for yrs_back, weight in zip([1,2,3],[4,2,1]):  # match training scheme: only prior seasons
            tmp = allpro_df.copy()
            tmp['season'] = tmp['Year'] + yrs_back
            tmp['weight'] = weight
            allpro_hist.append(tmp)
        allpro_wh = pd.concat(allpro_hist).drop_duplicates(['Player', 'season'])
        # Normalize names on both sides before joining (strips Jr./Sr./II/accents)
        allpro_wh = allpro_wh.copy()
        allpro_wh['_name_norm'] = allpro_wh['Player'].map(_norm_name)
        inj_all['_name_norm']   = inj_all['full_name'].map(_norm_name)
        inj_all   = inj_all.merge(allpro_wh[['_name_norm','season','weight']], on=['_name_norm','season'], how='left')
        inj_all   = inj_all[inj_all['weight'].notnull()]
        inj_all['weight'] = inj_all['weight'] * inj_all['_status_wt']  # scale by injury severity
        inj_wt    = inj_all.groupby(['season','week','team'])['weight'].sum().reset_index().rename(columns={'weight':'inj_ap_wt'})
        upcoming  = upcoming.merge(inj_wt.rename(columns={'team':'home_team'}).drop(columns=['season','week']), on='home_team', how='left').rename(columns={'inj_ap_wt':'home_inj_ap_wt'})
        upcoming  = upcoming.merge(inj_wt.rename(columns={'team':'away_team'}).drop(columns=['season','week']), on='away_team', how='left').rename(columns={'inj_ap_wt':'away_inj_ap_wt'})
        upcoming[['home_inj_ap_wt','away_inj_ap_wt']] = upcoming[['home_inj_ap_wt','away_inj_ap_wt']].fillna(0)
        upcoming['home_active_allpro_weighted'] = (upcoming['home_allpro_last_3_years_weighted'] - upcoming['home_inj_ap_wt']).clip(lower=0)
        upcoming['away_active_allpro_weighted'] = (upcoming['away_allpro_last_3_years_weighted'] - upcoming['away_inj_ap_wt']).clip(lower=0)
        upcoming['diff_active_allpro_weighted'] = upcoming['home_active_allpro_weighted'] - upcoming['away_active_allpro_weighted']
        # Approximation: cannot retroactively adjust prior-year AllPro counts for current-week injuries;
        # diff_allpro_prev_year uses the raw prior-year count without injury status correction.
        upcoming['diff_active_allpro_prev_year']= upcoming['diff_allpro_prev_year']

    except Exception as e:
        print(f"  ⚠️  Injury data unavailable: {e} — using zeros")
        for col in ['home_injured_count','away_injured_count','diff_injured_count','diff_active_allpro_weighted','diff_active_allpro_prev_year']:
            upcoming[col] = 0

    # ── Group 10 — coach win pct ─────────────────────────────────────────────
    if coach_hist_df is None:
        _raw_coach = nfl.load_schedules(list(range(1999, target_season + 1)))
        _ch = _raw_coach.to_pandas() if hasattr(_raw_coach, 'to_pandas') else pd.DataFrame(_raw_coach)
        coach_hist = _ch[_ch['result'].notna()].copy()
    else:
        coach_hist = coach_hist_df
    home_c = coach_hist[['game_id','season','week','home_team','away_team','home_score','away_score','home_coach']].copy()
    home_c.rename(columns={'home_team':'team','away_team':'opponent','home_score':'team_score','away_score':'opponent_score','home_coach':'coach'}, inplace=True)
    away_c = coach_hist[['game_id','season','week','away_team','home_team','away_score','home_score','away_coach']].copy()
    away_c.rename(columns={'away_team':'team','home_team':'opponent','away_score':'team_score','home_score':'opponent_score','away_coach':'coach'}, inplace=True)
    games_df = pd.concat([home_c, away_c], ignore_index=True)
    games_df['win'] = (games_df['team_score'] > games_df['opponent_score']).astype(int)

    def cumulative_coach(group):
        group = group.sort_values(['season','week','game_id']).copy()
        group['cumulative_wins']  = group['win'].cumsum().shift(fill_value=0)
        group['cumulative_games'] = group['win'].expanding().count().shift(fill_value=0)
        return group

    games_df = games_df.groupby('coach', group_keys=False).apply(cumulative_coach)
    games_df['coach_win_pct_prior'] = (
        games_df['cumulative_wins'] / games_df['cumulative_games'].replace(0, np.nan)
    ).fillna(0).round(3)
    latest_coach_wp = (
        games_df.sort_values(['season', 'week', 'game_id'])
        .groupby('coach').last().reset_index()[['coach', 'coach_win_pct_prior']]
    )
    upcoming = upcoming.merge(
        latest_coach_wp.rename(columns={'coach': 'home_coach', 'coach_win_pct_prior': 'home_coach_win_pct_prior'}),
        on='home_coach', how='left'
    )
    upcoming = upcoming.merge(
        latest_coach_wp.rename(columns={'coach': 'away_coach', 'coach_win_pct_prior': 'away_coach_win_pct_prior'}),
        on='away_coach', how='left'
    )
    _league_coach_avg = latest_coach_wp['coach_win_pct_prior'].mean() if len(latest_coach_wp) > 0 else 0.5
    upcoming[['home_coach_win_pct_prior', 'away_coach_win_pct_prior']] = (
        upcoming[['home_coach_win_pct_prior', 'away_coach_win_pct_prior']].fillna(_league_coach_avg)
    )

    # ── Final check ──────────────────────────────────────────────────────────
    _all_required = list(dict.fromkeys(model_features + ens_feat_cols + lgbm_feat_cols))
    missing = [f for f in _all_required if f not in upcoming.columns]
    if missing:
        print(f"  ⚠️  {len(missing)} features missing for week {target_week}: {missing}")
        for m in missing:
            upcoming[m] = 0

    _nan_cols = [c for c in upcoming.select_dtypes(include='number').columns
                 if upcoming[c].isna().any()]
    if _nan_cols:
        print(f"  ⚠️  {len(_nan_cols)} NaN column(s) imputed with median: {_nan_cols}")
    upcoming = upcoming.fillna(upcoming.median(numeric_only=True))
    return upcoming



# ── Numeric feature builder (Ensemble and LightGBM) ─────────────────────────
def build_numeric_features(upcoming_df, feature_cols, enc):
    """Build ordinal-encoded numeric feature matrix for Ensemble and LightGBM."""
    df = upcoming_df.copy()
    if hasattr(enc, 'categories_'):
        for i, col in enumerate(["roof", "surface"]):
            known    = set(enc.categories_[i])
            fallback = enc.categories_[i][0]
            df[col]  = df[col].fillna(fallback).apply(lambda v: v if v in known else fallback)
    df[["roof", "surface"]] = enc.transform(df[["roof", "surface"]])
    X = np.zeros((len(df), len(feature_cols)), dtype="float32")
    for i, col in enumerate(feature_cols):
        if col in df.columns:
            X[:, i] = pd.to_numeric(df[col], errors="coerce").fillna(0).values
    return X

# ── Run predictions ───────────────────────────────────────────────────────────
def run_predictions(target_week, target_season, full_schedule, pbp_rp, allpro_df, week_margin_lkp=None, coach_hist_df=None):
    print(f"Building features for season {target_season} week {target_week}...")
    upcoming = build_features(target_week, target_season, full_schedule, pbp_rp, allpro_df, week_margin_lkp=week_margin_lkp, coach_hist_df=coach_hist_df)
    if upcoming is None or upcoming.empty:
        print("No games found.")
        return None

    # ── XGBoost (prod) ───────────────────────────────────────────────────────
    X     = upcoming[model_features].copy()
    preds = pipeline.predict(X)

    # ── Ensemble (fixed75: 0.75 XGB + 0.25 Ridge) ────────────────────────────
    X_ens_raw = build_numeric_features(upcoming, ens_feat_cols, ens_enc)
    X_ens_sc  = ens_scaler.transform(X_ens_raw)
    ens_preds = (ens_xgb_weight * ens_xgb.predict(X_ens_raw)
                 + (1 - ens_xgb_weight) * ens_ridge.predict(X_ens_sc))

    # ── Ridge (extracted from ensemble pkg) ─────────────────────────────────────
    ridge_preds = ens_ridge.predict(X_ens_sc)

    # ── LightGBM (independent voter) ─────────────────────────────────────────
    X_lgbm    = build_numeric_features(upcoming, lgbm_feat_cols, ens_enc)
    lgbm_preds = lgbm_model.predict(X_lgbm)

    results = upcoming[['game_id','home_team','away_team','gameday','spread_line']].copy()
    spread  = results['spread_line']

    results['predicted_margin']     = preds.round(1)
    results['model_edge']           = (results['predicted_margin'] - spread).round(1)
    results['ens_predicted_margin']   = ens_preds.round(1)
    results['ens_model_edge']         = (results['ens_predicted_margin'] - spread).round(1)
    results['ridge_predicted_margin'] = ridge_preds.round(1)
    results['ridge_model_edge']       = (results['ridge_predicted_margin'] - spread).round(1)
    results['lgbm_predicted_margin']  = lgbm_preds.round(1)
    results['lgbm_model_edge']        = (results['lgbm_predicted_margin'] - spread).round(1)

    def side(edge, home, away):
        if edge > 0:  return f"HOME ({home})"
        if edge < 0:  return f"AWAY ({away})"
        return "PASS"

    results['recommendation']     = results.apply(lambda r: side(r['model_edge'],     r['home_team'], r['away_team']), axis=1)
    results['ens_recommendation']   = results.apply(lambda r: side(r['ens_model_edge'],   r['home_team'], r['away_team']), axis=1)
    results['ridge_recommendation'] = results.apply(lambda r: side(r['ridge_model_edge'], r['home_team'], r['away_team']), axis=1)
    results['lgbm_recommendation']  = results.apply(lambda r: side(r['lgbm_model_edge'],  r['home_team'], r['away_team']), axis=1)

    # Consensus tier: HIGH = XGB (standalone)/Ridge/LightGBM all agree direction + abs(ens_model_edge) >= 3pt
    def consensus_tier(row):
        sides = [row['recommendation'], row['ridge_recommendation'], row['lgbm_recommendation']]
        agree = all(s != 'PASS' for s in sides) and len(set(sides)) == 1
        edge  = abs(row['ens_model_edge'])
        if agree and edge >= 3: return 'HIGH'
        if agree and edge >= 1: return 'MEDIUM'
        return 'PASS'
    results['consensus_tier'] = results.apply(consensus_tier, axis=1)

    results = results.sort_values('ens_model_edge', key=abs, ascending=False)
    display_cols = ['home_team','away_team','spread_line','ens_model_edge','ridge_model_edge','model_edge','lgbm_model_edge','consensus_tier']
    print(results[display_cols].to_string(index=False))
    return results


# ── Update results ────────────────────────────────────────────────────────────
def update_results(season, week):
    if not os.path.exists(TRACKER_PATH):
        print("No tracker found — skipping results update")
        return
    print(f"Updating results for season {season} week {week}...")
    tracker = pd.read_csv(TRACKER_PATH)
    raw     = nfl.load_schedules([season])
    sched   = raw.to_pandas() if hasattr(raw, 'to_pandas') else pd.DataFrame(raw)

    # Pull result AND individual scores
    actual  = sched[(sched['season'] == season) & (sched['week'] == week)][
        ['game_id', 'result', 'home_score', 'away_score']
    ].rename(columns={'result': 'actual_margin'})

    if actual['actual_margin'].isna().all():
        print(f"Results not yet available for week {week} — skipping")
        return
    mask    = (tracker['season'] == season) & (tracker['week'] == week)
    indices = tracker[mask].index
    if len(indices) == 0:
        print(f"No predictions found for week {week} — skipping")
        return
    rows = tracker.loc[indices].copy()
    rows = rows.merge(actual, on='game_id', how='left', suffixes=('_old', '_new'))
    rows['actual_margin'] = rows['actual_margin_new']
    rows['home_score']    = rows['home_score_new']
    rows['away_score']    = rows['away_score_new']
    rows = rows.drop(columns=['actual_margin_old', 'actual_margin_new', 'home_score_old', 'home_score_new', 'away_score_old', 'away_score_new'], errors='ignore')
    def _home_covered(margin, spread):
        if pd.isna(margin) or pd.isna(spread):
            return float('nan')
        if margin == spread:  # push — no ATS result
            return float('nan')
        return float(margin > spread)
    rows['home_covered'] = rows.apply(
        lambda r: _home_covered(r['actual_margin'], r['spread_line']), axis=1
    )
    def _score(edge, covered):
        if pd.isna(edge) or pd.isna(covered) or edge == 0:
            return float('nan')
        return int((edge > 0) == (covered == 1))

    rows['model_correct']     = rows.apply(lambda r: _score(r['model_edge'],     r['home_covered']), axis=1)
    if 'ens_model_edge' in rows.columns:
        rows['ens_model_correct'] = rows.apply(lambda r: _score(r['ens_model_edge'],   r['home_covered']), axis=1)
    if 'ridge_model_edge' in rows.columns:
        rows['ridge_model_correct'] = rows.apply(lambda r: _score(r['ridge_model_edge'], r['home_covered']), axis=1)
    if 'lgbm_model_edge' in rows.columns:
        rows['lgbm_model_correct'] = rows.apply(lambda r: _score(r['lgbm_model_edge'],  r['home_covered']), axis=1)

    tracker.loc[indices, 'actual_margin'] = rows['actual_margin'].values
    tracker.loc[indices, 'home_covered']  = rows['home_covered'].values
    tracker.loc[indices, 'model_correct'] = rows['model_correct'].values
    tracker.loc[indices, 'home_score']    = rows['home_score'].values
    tracker.loc[indices, 'away_score']    = rows['away_score'].values
    for col in ['ens_model_correct', 'ridge_model_correct', 'lgbm_model_correct']:
        if col in rows.columns:
            tracker.loc[indices, col] = rows[col].values
    tracker.to_csv(TRACKER_PATH, index=False)

    correct = int(rows['model_correct'].sum())
    total   = int(rows['model_correct'].notna().sum())
    if total > 0:
        print(f"✅ Week {week} ATS: {correct}/{total} ({correct/total*100:.1f}%)")
    else:
        print(f"✅ Week {week}: results updated (no model-predicted games)")


# ── Log predictions ───────────────────────────────────────────────────────────
def log_predictions(results_df, season, week, mode):
    extra = [c for c in ['ens_predicted_margin','ens_model_edge','ens_recommendation',
                          'ridge_predicted_margin','ridge_model_edge','ridge_recommendation',
                          'lgbm_predicted_margin','lgbm_model_edge','lgbm_recommendation',
                          'consensus_tier'] if c in results_df.columns]
    log = results_df[['game_id','home_team','away_team','gameday','spread_line',
                       'predicted_margin','model_edge','recommendation'] + extra].copy()
    log['season']        = season
    log['week']          = week
    log['mode']          = mode
    log['logged_at']     = datetime.now().strftime('%Y-%m-%d %H:%M')
    log['actual_margin'] = None
    log['home_covered']  = None
    log['model_correct'] = None
    log['home_score']    = None
    log['away_score']    = None
    if os.path.exists(TRACKER_PATH):
        tracker = pd.read_csv(TRACKER_PATH)
        mask    = (tracker['season'] == season) & (tracker['week'] == week)
        if mask.any():
            print(f"Replacing existing week {week} predictions ({mode} refresh)...")
            _res_cols = ['actual_margin', 'home_covered', 'model_correct', 'home_score', 'away_score',
                         'ens_model_correct', 'ridge_model_correct', 'lgbm_model_correct']
            old_results = tracker.loc[mask, ['game_id'] + [c for c in _res_cols if c in tracker.columns]].copy()
            tracker = tracker[~mask]
            log = log.merge(
                old_results.rename(columns={c: f'_old_{c}' for c in _res_cols}),
                on='game_id', how='left'
            )
            for col in _res_cols:
                old_col = f'_old_{col}'
                if old_col in log.columns:
                    log[col] = log[old_col]
                    log = log.drop(columns=[old_col])
        updated = pd.concat([tracker, log], ignore_index=True)
        updated.to_csv(TRACKER_PATH, index=False)
    else:
        log.to_csv(TRACKER_PATH, index=False)
    print(f"✅ Week {week} predictions saved ({mode} — {len(log)} games)")


# ── Main ──────────────────────────────────────────────────────────────────────

## Run Predictions

Loads the full season schedule and two seasons of PBP data, then dispatches to the appropriate mode. Both PBP seasons feed Groups 2 and 6 rolling features (providing prior-season history for early weeks); the prior season is also used independently in Group 8 for passer rating.

| Mode | Trigger | Behaviour |
|------|---------|----------|
| `tuesday` | Tue 9am ET | Updates previous week's results → runs new predictions → logs to tracker |
| `thursday` | Thu 9pm ET | Refreshes predictions with latest injury data → overwrites week's entry in tracker |
| `sunday` | Sun 9am ET | Final predictions before kickoff → overwrites week's entry in tracker |
| `backfill` | Manual | Runs predictions for `TARGET_WEEK` → logs → immediately fills in actual results |

All modes call `run_predictions()` which builds the 77-feature matrix, runs Ensemble fixed75 (edge-setter), XGBoost, Ridge, and LightGBM, then sorts games by absolute ensemble edge. Results are written to `betting/predictions_tracker.csv`.


In [ ]:
# Auto-detect season from current date if not overridden
if TARGET_SEASON is None:
    _now = datetime.now()
    TARGET_SEASON = _now.year if _now.month >= 9 else _now.year - 1
    print(f"Auto-detected TARGET_SEASON={TARGET_SEASON}")

# Load all schedules in one call (1999–present) — provides full_schedule,
# coach history, and week-margin lookup without duplicate API fetches
print("Loading schedules (1999–present, single pass)...")
_raw_all   = nfl.load_schedules(list(range(1999, TARGET_SEASON + 1)))
_all_sched = _raw_all.to_pandas() if hasattr(_raw_all, 'to_pandas') else pd.DataFrame(_raw_all)
_all_sched['season'] = _all_sched['season'].astype(int)
_all_sched['week']   = _all_sched['week'].astype(int)

full_schedule = _all_sched[_all_sched['season'] == TARGET_SEASON].copy().reset_index(drop=True)
coach_hist_df = _all_sched[_all_sched['result'].notna()].copy()
_hist_df_lkp  = _all_sched[
    (_all_sched['season'] >= 2014) & (_all_sched['season'] < TARGET_SEASON) &
    (_all_sched['game_type'] == 'REG') & _all_sched['result'].notna()
]
week_margin_lkp = _hist_df_lkp.groupby('week')['result'].apply(lambda x: x.abs().mean())
print(f"Schedules loaded: {len(full_schedule)} current-season games | "
      f"{len(coach_hist_df)} completed (coach) | {len(week_margin_lkp)} week-margin keys")

if TARGET_WEEK is None:
    TARGET_WEEK, PREV_WEEK = get_week_info(TARGET_SEASON, schedule_df=full_schedule)
    if TARGET_WEEK is None:
        raise ValueError("Season is over — no predictions to run. See you in September!")
else:
    TARGET_WEEK = int(TARGET_WEEK)
    PREV_WEEK   = (TARGET_WEEK - 1) if TARGET_WEEK > 1 else None

print(f"Upcoming week: {TARGET_WEEK} | Previous week: {PREV_WEEK}")

print("Loading PBP data (this takes ~60s)...")
raw_pbp = nfl.load_pbp([TARGET_SEASON, TARGET_SEASON - 1])
pbp     = raw_pbp.to_pandas() if hasattr(raw_pbp, 'to_pandas') else pd.DataFrame(raw_pbp)
pbp_rp  = pbp[
    pbp['play_type'].isin(['run','pass']) &
    pbp['posteam'].notna() &
    pbp['defteam'].notna()
].copy()
print(f"PBP loaded: {pbp_rp.shape} | Seasons: {sorted(pbp_rp['season'].unique())}")

if MODE == 'tuesday':
    if PREV_WEEK:
        update_results(TARGET_SEASON, PREV_WEEK)
    if TARGET_WEEK:
        results = run_predictions(TARGET_WEEK, TARGET_SEASON, full_schedule, pbp_rp, allpro_df, week_margin_lkp=week_margin_lkp, coach_hist_df=coach_hist_df)
        if results is not None:
            log_predictions(results, TARGET_SEASON, TARGET_WEEK, mode='tuesday')

# thursday and sunday run the same prediction logic;
# the only difference is when in the week they execute (injury data freshness).
elif MODE == 'thursday':
    if TARGET_WEEK:
        results = run_predictions(TARGET_WEEK, TARGET_SEASON, full_schedule, pbp_rp, allpro_df, week_margin_lkp=week_margin_lkp, coach_hist_df=coach_hist_df)
        if results is not None:
            log_predictions(results, TARGET_SEASON, TARGET_WEEK, mode='thursday')

elif MODE == 'sunday':
    if TARGET_WEEK:
        results = run_predictions(TARGET_WEEK, TARGET_SEASON, full_schedule, pbp_rp, allpro_df, week_margin_lkp=week_margin_lkp, coach_hist_df=coach_hist_df)
        if results is not None:
            log_predictions(results, TARGET_SEASON, TARGET_WEEK, mode='sunday')

elif MODE == 'backfill':
    if TARGET_WEEK:
        results = run_predictions(TARGET_WEEK, TARGET_SEASON, full_schedule, pbp_rp, allpro_df, week_margin_lkp=week_margin_lkp, coach_hist_df=coach_hist_df)
        if results is not None:
            log_predictions(results, TARGET_SEASON, TARGET_WEEK, mode='backfill')
            update_results(TARGET_SEASON, TARGET_WEEK)

else:
    print(f"Unknown mode: {MODE}. Use tuesday, thursday, sunday, or backfill.")
